# CakraWatch — Fine-Tune IndoBERT untuk Deteksi Hoaks Bahasa Indonesia
### Kartika Cyber Hackathon 2026 · Kategori 04 Siber Sosial

Notebook ini melatih model **IndoBERT** (`indolem/indobert-base-uncased`) untuk mengklasifikasikan teks berita/klaim Bahasa Indonesia menjadi **HOAKS** atau **VALID**, lengkap dengan skor keyakinan dan penjelasan frasa pemicu.

**Target yang harus dikalahkan** (baseline TF-IDF + Logistic Regression, sudah terukur):
- Accuracy 0,9375 · F1 HOAKS 0,947 · F1 VALID 0,924 · **Macro F1 0,9354**

**Cara pakai:** jalankan sel dari atas ke bawah (Runtime → Run all). Pastikan runtime GPU aktif (Runtime → Change runtime type → T4 GPU).

## Langkah 1 — Cek GPU dan install library

In [ ]:
import torch
print("GPU tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PERINGATAN: GPU tidak aktif. Ubah dulu: Runtime > Change runtime type > T4 GPU, lalu jalankan ulang sel ini.")

In [ ]:
# Versi library disematkan agar notebook stabil
!pip install -q "transformers==4.46.3" "accelerate==1.1.1" scikit-learn
print("Install selesai.")

## Langkah 2 — Unduh dan periksa dataset

Dataset inti: repo GitHub `JibranFawaid/turnbackhoax-dataset` (data TurnBackHoax berlabel).
- Label `0` = HOAKS (683 data: headline dan isi klaim hoaks)
- Label `1` = VALID (433 data: artikel klarifikasi resmi)

File dipisah tanda titik koma dan berencoding latin-1.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/JibranFawaid/turnbackhoax-dataset/main/Scrapping.csv"
try:
    df = pd.read_csv(URL, sep=";", encoding="latin-1", on_bad_lines="skip")
    print("Unduh dari GitHub berhasil.")
except Exception as e:
    print("Unduh gagal:", e)
    print("Jalankan sel cadangan di bawah untuk upload file manual.")
    df = None

if df is not None:
    df.columns = [c.strip() for c in df.columns]
    df = df[["Label", "Headline", "Body"]].dropna()
    df = df[df["Label"].isin([0, 1])]
    print("Jumlah data bersih:", len(df))
    print(df["Label"].value_counts().rename({0: "HOAKS", 1: "VALID"}))
    display(df.head(3))

In [ ]:
# SEL CADANGAN — jalankan hanya jika unduh otomatis gagal.
# Upload file Scrapping.csv dari laptopmu.
# from google.colab import files
# up = files.upload()
# import io
# df = pd.read_csv(io.BytesIO(up["Scrapping.csv"]), sep=";", encoding="latin-1", on_bad_lines="skip")
# df.columns = [c.strip() for c in df.columns]
# df = df[["Label", "Headline", "Body"]].dropna()
# df = df[df["Label"].isin([0, 1])]
# print("Jumlah data:", len(df))

## Langkah 3 — Siapkan teks dan bagi data

Teks yang dilatih = Headline + isi Body. Data dibagi 80% latih / 20% uji dengan stratifikasi (proporsi label dijaga sama) dan seed tetap agar hasil bisa direproduksi.

In [ ]:
from sklearn.model_selection import train_test_split

df["text"] = (df["Headline"].astype(str) + ". " + df["Body"].astype(str)).str.strip()
df["label"] = df["Label"].astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
print("Latih:", len(train_df), "| Uji:", len(test_df))
print("Distribusi latih:\n", train_df["label"].value_counts().rename({0: "HOAKS", 1: "VALID"}))
print("Distribusi uji:\n", test_df["label"].value_counts().rename({0: "HOAKS", 1: "VALID"}))

## Langkah 4 — Tokenisasi IndoBERT

Panjang maksimum 256 token (judul + potongan isi). Kelas VALID lebih sedikit, jadi pada Langkah 5 kerugian (loss) akan diberi bobot kelas agar model tidak malas menebak kelas mayoritas.

In [ ]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

MODEL_NAME = "indolem/indobert-base-uncased"
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HoaxDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=MAX_LEN, return_tensors=None
        )
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = HoaxDataset(train_df["text"], train_df["label"])
test_ds = HoaxDataset(test_df["text"], test_df["label"])
print("Dataset siap. Contoh token:", train_ds[0]["input_ids"][:10])

## Langkah 5 — Fine-tune

Konfigurasi mengikuti literatur IndoBERT untuk hoaks Indonesia: learning rate 5e-5, batch 16, pemanasan 10%, evaluasi tiap epoch, model terbaik dipilih dari **macro F1** tertinggi. Bobot kelas dihitung otomatis dari distribusi data latih.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# Bobot kelas: kelas yang lebih sedikit diberi bobot lebih besar
counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df) / (2 * counts[0]), len(train_df) / (2 * counts[1])],
    dtype=torch.float
)
print("Bobot kelas [HOAKS, VALID]:", class_weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "HOAKS", 1: "VALID"}, label2id={"HOAKS": 0, "VALID": 1}
)

args = TrainingArguments(
    output_dir="/content/cakrawatch_out",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    seed=42,
    report_to="none",
    logging_steps=25,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
print("Training selesai.")

## Langkah 6 — Evaluasi akhir dan bandingkan dengan baseline

Catatan kejujuran data: kelas VALID pada dataset ini berbentuk artikel klarifikasi pemeriksa fakta. Model bisa saja sebagian belajar dari gaya bahasa klarifikasi. Untuk laporan, ini dicatat sebagai keterbatasan data dan alasan roadmap pengumpulan berita valid dari media arus utama.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds_out = trainer.predict(test_ds)
preds = np.argmax(preds_out.predictions, axis=1)
print(classification_report(test_df["label"], preds, target_names=["HOAKS", "VALID"]))
print("Confusion matrix [baris = aktual, kolom = prediksi]:")
print(confusion_matrix(test_df["label"], preds))
acc = accuracy_score(test_df["label"], preds)
mf1 = f1_score(test_df["label"], preds, average="macro")
print(f"\nAccuracy IndoBERT : {acc:.4f}   (baseline TF-IDF: 0.9375)")
print(f"Macro F1 IndoBERT  : {mf1:.4f}   (baseline TF-IDF: 0.9354)")
print("Status:", "MENGALAHKAN baseline" if mf1 > 0.9354 else "belum mengalahkan baseline, lihat catatan di Langkah 8")

## Langkah 7 — Uji prediksi + penjelasan frasa pemicu

Penjelasan memakai metode **occlusion**: setiap kata dicabut satu per satu dan penurunan probabilitas HOAKS-nya diukur. Kata yang paling menurunkan skor saat dicabut adalah frasa pemicu utamanya. Ini yang nanti tampil sebagai sorotan di dashboard CakraWatch.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device); model.eval()

def predict_proba(text):
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=1)[0]
    return probs[0].item(), probs[1].item()  # (P HOAKS, P VALID)

def analyze(text, top_k=5):
    p_hoax, p_valid = predict_proba(text)
    label = "HOAKS" if p_hoax >= p_valid else "VALID"
    conf = max(p_hoax, p_valid)
    print(f"Teks   : {text[:120]}...")
    print(f"Vonis  : {label}  (keyakinan {conf:.1%})  |  P(HOAKS)={p_hoax:.3f}  P(VALID)={p_valid:.3f}")
    words = text.split()
    if len(words) > 40:
        print("(Penjelasan frasa pemicu dibatasi 40 kata pertama)")
        words = words[:40]
    drops = []
    for i, w in enumerate(words):
        reduced = " ".join(words[:i] + words[i+1:])
        if not reduced:
            continue
        ph, _ = predict_proba(reduced)
        drops.append((words[i], p_hoax - ph))
    drops.sort(key=lambda x: x[1], reverse=True)
    print("Frasa pemicu terkuat (paling menaikkan skor HOAKS):")
    for w, d in drops[:top_k]:
        if d > 0:
            print(f"  - '{w}'  (kontribusi +{d:.3f})")
    print()
    return label, conf

# Contoh uji 1: gaya klaim viral berantai
analyze("SEBARKAN! Mulai besok semua ATM di Indonesia berhenti beroperasi selamanya. Segera tarik semua uang tunai Anda sebelum rekening dibekukan pemerintah! Jangan sampai terlambat, kirim ke semua kontak Anda sekarang juga!!!")

# Contoh uji 2: gaya laporan media arus utama
analyze("Bank Indonesia menyatakan layanan ATM dan perbankan elektronik tetap beroperasi normal. Masyarakat diimbau tidak panik dan memverifikasi informasi melalui kanal resmi sebelum membagikannya.")

## Langkah 8 — Simpan model

Model terbaik (berdasarkan macro F1) disimpan lalu diunduh sebagai zip. File ini yang nanti dimuat dashboard demo CakraWatch dan dilampirkan sebagai bukti di repo submission.

In [ ]:
trainer.save_model("/content/cakrawatch_indobert_final")
tokenizer.save_pretrained("/content/cakrawatch_indobert_final")
print("Model tersimpan di /content/cakrawatch_indobert_final")

# Unduh sebagai zip (opsional, ukuran sekitar 500 MB)
# !cd /content && zip -qr cakrawatch_indobert_final.zip cakrawatch_indobert_final
# from google.colab import files; files.download("/content/cakrawatch_indobert_final.zip")

## Catatan strategi bila hasil belum mengalahkan baseline

1. Naikkan epoch ke 10 dan/atau turunkan learning rate ke 3e-5.
2. Tambah data hoaks 2024 (dataset raihankalla) HANYA bila diimbangi kelas valid sepadan atau class weighting dipertahankan, karena augmentasi naif terbukti menjatuhkan F1 VALID ke 0,78.
3. Kumpulkan berita valid asli dari media arus utama (bukan hanya artikel klarifikasi) agar model belajar isi berita, bukan gaya bahasa pemeriksa fakta.
4. Untuk demo, kecepatan dan penjelasan frasa pemicu sering lebih memukau juri daripada selisih akurasi 1-2 persen. Pertahankan occlusion sebagai fitur penjelasan utama.

Selesai. Kirim tangkapan layar hasil Langkah 6 dan 7 ke Muse untuk dicatat di goal dan dibahas langkah dashboard-nya.